# Hierarchical Feature Index for MuData — Proof of Concept

This notebook demonstrates a proof-of-concept implementation of a **hierarchical/tree-like feature index** for [MuData](https://mudata.readthedocs.io/), designed for the hierarchical quantification structure in MS-based proteomics.

**Problem**: In LC/MS proteomics, precursors and proteins have an N:M relationship. MuData can store both modalities side-by-side, but provides no way to *query* the mapping between them.

**Solution**: We introduce `FeatureMapping` (a sparse bipartite graph between feature sets) and `HierarchicalMuData` (a MuData wrapper with cross-modality feature queries).

Related: [scverse/mudata#111](https://github.com/scverse/mudata/issues/111)

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import FeatureMapping, HierarchicalMuData, make_synthetic_proteomics_data

## 1. Create synthetic proteomics data

We generate a dataset with:
- 6 samples (LC/MS runs)
- 200 proteins (with ~9 precursors each → ~1800 precursors)
- ~5% of precursors shared between proteins (N:M mapping)
- 10% missing values (typical for DIA proteomics)

In [ ]:
hmdata = make_synthetic_proteomics_data(
    n_samples=6,
    n_proteins=200,
    n_precursors_per_protein=(3, 15),
    shared_precursor_fraction=0.05,
    missing_value_rate=0.1,
    seed=42,
)
hmdata

In [ ]:
hmdata.mapping_summary()

## 2. Inspect the underlying structure

Under the hood, `HierarchicalMuData` wraps a standard `MuData` object. The modalities are regular `AnnData` objects:

In [ ]:
print("MuData modalities:")
for name, adata in hmdata.mod.items():
    print(f"  {name}: {adata.n_obs} samples × {adata.n_vars} features")

print(f"\nRegistered mappings: {list(hmdata.mappings.keys())}")

The `FeatureMapping` stores the N:M relationship as a sparse boolean adjacency matrix:

In [ ]:
mapping = hmdata.get_mapping("precursors", "proteins")
print(mapping)
print(f"\nAdjacency matrix: {mapping.adjacency.shape}, density = {mapping.adjacency.nnz / np.prod(mapping.adjacency.shape):.4f}")
print(f"\nEdges sample:")
mapping.edges.head(10)

## 3. Cross-modality queries

The key new capability: querying features across the hierarchy.

### Forward: precursor → protein

In [ ]:
# Pick a precursor that maps to multiple proteins (shared peptide)
multi_mapped = mapping.edges.groupby("source").size()
shared_prec = multi_mapped[multi_mapped > 1].index[0]

proteins = hmdata.get_related_features("precursors", "proteins", shared_prec)
print(f"Precursor '{shared_prec}' maps to {len(proteins)} proteins: {proteins}")

### Reverse: protein → precursors

In [ ]:
prot = "PROT_0042"
precursors = hmdata.get_related_features("proteins", "precursors", prot)
print(f"Protein '{prot}' has {len(precursors)} precursors: {precursors}")

### Data slicing: get precursor intensities for a protein

This is the query from the [mudata issue #111](https://github.com/scverse/mudata/issues/111):
> *"Plot a boxplot of the intensity distributions of precursors that map to the protein MAPK"*

In [ ]:
prec_slice = hmdata.slice_data("precursors", by="proteins", features=prot)
print(f"Sliced precursor AnnData for {prot}: {prec_slice.shape}")
prec_slice.to_df()

## 4. Application: intensity distribution across hierarchy

A common analysis task: compare the precursor-level intensity distributions
that contribute to a given protein.

In [ ]:
target_prot = "PROT_0010"
prec_data = hmdata.slice_data("precursors", by="proteins", features=target_prot)
df = prec_data.to_df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot of precursor intensities
ax = axes[0]
df_log = np.log10(df.replace(0, np.nan))
df_log.boxplot(ax=ax, rot=45)
ax.set_title(f"Precursor intensity distributions for {target_prot}")
ax.set_ylabel("log10(intensity)")
ax.set_xlabel("Precursor")

# Compare protein vs precursor intensities
ax = axes[1]
prot_vals = hmdata["proteins"][:, target_prot].X.flatten()
prec_vals = prec_data.X.flatten()
prec_vals = prec_vals[~np.isnan(prec_vals)]

ax.hist(np.log10(prec_vals), bins=20, alpha=0.7, label="Precursors", density=True)
ax.axvline(np.log10(np.nanmedian(prot_vals)), color="red", ls="--", lw=2, label=f"{target_prot} (protein)")
ax.set_xlabel("log10(intensity)")
ax.set_ylabel("Density")
ax.set_title(f"Precursor vs protein intensity for {target_prot}")
ax.legend()

plt.tight_layout()
plt.show()

## 5. Mapping topology: visualize the N:M structure

Let's look at the degree distribution — how many proteins does each precursor map to, and vice versa?

In [ ]:
adj = mapping.adjacency

prec_degree = np.asarray(adj.sum(axis=1)).flatten()  # proteins per precursor
prot_degree = np.asarray(adj.sum(axis=0)).flatten()  # precursors per protein

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(prec_degree, bins=range(1, int(prec_degree.max()) + 2), edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Proteins per precursor")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Precursor → protein degree (N:M)\n{(prec_degree > 1).sum()} shared precursors")

axes[1].hist(prot_degree, bins=20, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Precursors per protein")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Protein → precursor degree\nmean={prot_degree.mean():.1f}, range=[{prot_degree.min()}, {prot_degree.max()}]")

plt.tight_layout()
plt.show()

## 6. Subgraph queries

Extract a sub-mapping for a specific set of proteins (e.g., a pathway):

In [ ]:
pathway_proteins = ["PROT_0001", "PROT_0010", "PROT_0042", "PROT_0100"]

sub = mapping.subgraph(targets=pathway_proteins)
print(sub)
print(f"\nEdges in sub-mapping:")
sub.edges

## 7. Serialization roundtrip

Mappings can be serialized into `MuData.uns` and reconstructed, so they survive disk I/O:

In [ ]:
hmdata.store_mappings_in_uns()
print("Stored in uns:", [k for k in hmdata.mdata.uns.keys() if k.startswith("mapping_")])

# Reconstruct from MuData
hmdata_restored = HierarchicalMuData.from_mudata(hmdata.mdata)
print("\nRestored:", hmdata_restored)

# Verify queries produce the same result
original = hmdata.get_related_features("proteins", "precursors", "PROT_0042")
restored = hmdata_restored.get_related_features("proteins", "precursors", "PROT_0042")
assert original == restored, "Roundtrip mismatch!"
print(f"\nRoundtrip verified: PROT_0042 → {len(original)} precursors ✓")

## 8. Multi-protein query

Query multiple proteins at once — useful for pathway-level analysis:

In [ ]:
multi_prots = ["PROT_0001", "PROT_0002", "PROT_0003"]
all_precs = hmdata.get_related_features("proteins", "precursors", multi_prots)
print(f"{len(multi_prots)} proteins → {len(all_precs)} unique precursors")

sliced = hmdata.slice_data("precursors", by="proteins", features=multi_prots)
print(f"Sliced AnnData: {sliced.shape}")

## Summary

This proof-of-concept demonstrates:

1. **`FeatureMapping`**: a sparse, bidirectional N:M mapping between feature sets
2. **`HierarchicalMuData`**: a MuData wrapper that supports cross-modality feature queries
3. **Key operations**:
   - Forward/reverse feature lookups (`get_related_features`)
   - Data slicing across the hierarchy (`slice_data`)
   - Subgraph extraction (`subgraph`)
   - Serialization via `MuData.uns` (`store_mappings_in_uns` / `from_mudata`)

### Next steps
- RFC formalizing the API and storage format
- Integration with real PSM readers (alphadia, DIA-NN) for automatic mapping construction
- Support for deeper hierarchies (fragments → precursors → peptides → proteins → genes)
- Discussion with mudata maintainers on whether this belongs in core or as an extension